# Dysgraphia Image Training (Kaggle Notebook)

This notebook trains a handwriting classifier for dysgraphia detection using transfer learning.

Dataset expected from Kaggle input:
- `bhdivyansh/dysgraphia-dataset`
- Classes: `PD` and `LPD`

Run this notebook in a **GPU-enabled Kaggle Notebook**.

In [ ]:
!nvidia-smi || true
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install compatible binary wheels for Kaggle Python 3.12.
# If you changed versions in this session, restart the kernel before continuing.
%pip -q install --upgrade --force-reinstall --no-cache-dir numpy==2.1.3 scikit-learn==1.5.2 timm==1.0.25 pyyaml pillow

In [ ]:
from pathlib import Path
import shutil
import random

# Kaggle mounts datasets under /kaggle/input/<dataset-slug>
INPUT_ROOT = Path('/kaggle/input/datasets/bhdivyansh/dysgraphia-dataset')
WORK_ROOT = Path('/kaggle/working/dysgraphia_image')
RAW_ROOT = WORK_ROOT / 'data' / 'raw'
SOURCE_DATASET = RAW_ROOT / 'source_dataset'
ARTIFACTS = WORK_ROOT / 'artifacts'

for p in [SOURCE_DATASET, ARTIFACTS / 'checkpoints', ARTIFACTS / 'exported', ARTIFACTS / 'reports']:
    p.mkdir(parents=True, exist_ok=True)

# Handle both direct and nested archive layouts.
candidate_roots = [INPUT_ROOT, INPUT_ROOT / 'Dysgraphia-dataset']
dataset_root = None
for c in candidate_roots:
    if (c / 'PD').exists() and (c / 'LPD').exists():
        dataset_root = c
        break

if dataset_root is None:
    raise FileNotFoundError('Could not find PD/LPD folders under /kaggle/input/dysgraphia-dataset')

def copy_dataset(src: Path, dst: Path):
    for item in dst.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)

copy_dataset(dataset_root, SOURCE_DATASET)
print('Dataset root:', dataset_root)
print('Copied to:', SOURCE_DATASET)
print('Top-level contents:', [p.name for p in SOURCE_DATASET.iterdir()])

In [ ]:
CLASS_MAP = {'PD': 'dysgraphia', 'LPD': 'typical'}

def list_images(folder: Path):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    return [p for p in folder.rglob('*') if p.suffix.lower() in exts]

def prepare_splits(source_root: Path, raw_root: Path, seed=42, train_ratio=0.7, val_ratio=0.15):
    random.seed(seed)

    for split in ['train', 'val', 'test']:
        for cls in CLASS_MAP.values():
            dst = raw_root / split / cls
            dst.mkdir(parents=True, exist_ok=True)
            for f in dst.glob('*'):
                if f.is_file():
                    f.unlink()

    summary = {}
    for src_cls, dst_cls in CLASS_MAP.items():
        files = list_images(source_root / src_cls)
        if not files:
            raise ValueError(f'No images found in {source_root / src_cls}')

        random.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        split_files = {
            'train': files[:n_train],
            'val': files[n_train:n_train+n_val],
            'test': files[n_train+n_val:],
        }

        for split, entries in split_files.items():
            dst_dir = raw_root / split / dst_cls
            for src in entries:
                dst = dst_dir / src.name
                if dst.exists():
                    dst = dst_dir / f'{src.stem}_{random.randint(1, 999999)}{src.suffix}'
                shutil.copy2(src, dst)

        summary[f'{dst_cls}_total'] = n
        summary[f'{dst_cls}_train'] = len(split_files['train'])
        summary[f'{dst_cls}_val'] = len(split_files['val'])
        summary[f'{dst_cls}_test'] = len(split_files['test'])

    return summary

summary = prepare_splits(SOURCE_DATASET, RAW_ROOT, seed=42)
summary

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Guard against stale interpreter state after pip installs in notebooks.
if tuple(int(x) for x in np.__version__.split('.')[:2]) < (2, 0):
    raise RuntimeError(
        f'NumPy {np.__version__} is loaded. Restart the Kaggle session and rerun from the top so binaries reload cleanly.'
    )

import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = 'convnextv2_base'

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE[0] + 24, IMG_SIZE[1] + 24)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=6),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(str(RAW_ROOT / 'train'), transform=train_tfms)
val_ds = datasets.ImageFolder(str(RAW_ROOT / 'val'), transform=eval_tfms)
test_ds = datasets.ImageFolder(str(RAW_ROOT / 'test'), transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Class mapping:', train_ds.class_to_idx)

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

In [ ]:
def run_epoch(loader, training=True):
    model.train(training)
    total_loss = 0.0
    labels_all = []
    preds_all = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(images)
                loss = loss_fn(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        labels_all.extend(labels.detach().cpu().tolist())
        preds_all.extend(preds.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(labels_all, preds_all, average='macro')
    acc = (torch.tensor(labels_all) == torch.tensor(preds_all)).float().mean().item()
    return {'loss': float(avg_loss), 'f1_macro': float(f1), 'accuracy': float(acc)}

best_val_f1 = -1.0
best_path = ARTIFACTS / 'checkpoints' / 'best_convnextv2_dysgraphia.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(train_loader, training=True)
    val_m = run_epoch(val_loader, training=False)
    scheduler.step()

    row = {'epoch': epoch, 'train': train_m, 'val': val_m}
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_m['loss']:.4f} train_f1={train_m['f1_macro']:.4f} | val_loss={val_m['loss']:.4f} val_f1={val_m['f1_macro']:.4f}")

    if val_m['f1_macro'] > best_val_f1:
        best_val_f1 = val_m['f1_macro']
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_to_idx': train_ds.class_to_idx,
            'model_name': MODEL_NAME,
            'image_size': IMG_SIZE,
        }, best_path)

(ARTIFACTS / 'reports').mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS / 'reports' / 'train_history.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

print('Best val F1:', best_val_f1)
print('Best checkpoint:', best_path)

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_labels = []
all_preds = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
target_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
report = classification_report(all_labels, all_preds, target_names=target_names, output_dict=True, zero_division=0)
conf = confusion_matrix(all_labels, all_preds).tolist()

with open(ARTIFACTS / 'reports' / 'test_classification_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
with open(ARTIFACTS / 'reports' / 'test_confusion_matrix.json', 'w', encoding='utf-8') as f:
    json.dump(conf, f, indent=2)

exported_path = ARTIFACTS / 'exported' / 'dysgraphia_image_model.pt'
torch.save(ckpt, exported_path)

print('Test accuracy:', report['accuracy'])
print('Test F1 macro:', report['macro avg']['f1-score'])
print('Exported model:', exported_path)

In [ ]:
# Optional: Save a single zip containing all artifacts
zip_path = shutil.make_archive('/kaggle/working/dysgraphia_image_artifacts', 'zip', ARTIFACTS)
print('Artifacts zip:', zip_path)
print('You can download it from the Kaggle Notebook output files panel.')